In [2]:
import requests
import json

# Define the API endpoint URL
api_url = "https://nileva.meta-bytes.net/api/products"

# Make GET request to the endpoint
response = requests.get(api_url)

# Check if request was successful
if response.status_code == 200:
    # Parse JSON response
    products_data = response.json()
    
    # Save data to file for later use
    with open('products.json', 'w') as f:
        json.dump(products_data, f, indent=4)
        
    print(f"Successfully retrieved {len(products_data)} products")
    print("Data saved to products.json")
else:
    print(f"Failed to get data: {response.status_code}")
    print(response.text)


Successfully retrieved 3 products
Data saved to products.json


In [3]:
with open(r'..\Dataset\products.json', 'r') as f:
    response = json.load(f)

response


{'event_type': 'products.updated',
 'timestamp': '2025-04-17T15:39:22.403774Z',
 'data': {'products': [{'product_id': 1,
    'name': 'Herforte Lotion Spray (to fill in gaps and treat alopecia)',
    'description': 'Hair forte is a mixture of natural extracts and extracts of the most important hair oils with anti-dandruff agent, to be a quick treatment for hair loss, guaranteed and tested by dermatologists for fortifying hair & increase hair shaft strength to prevent hair loss. Also, Hair forte spray stimulates hair germination to be an effective solution to fill the frontal spaces in women and men, in addition to treating dandruff & itching associated with dryness and inflammation of the scalp, Hair forte spray is suitable for all hair types even oily & dry scalp.\r\n\r\nComposition:-\r\nHair forte spray with its unique formula to be effective in stimulating the blood circulation of the scalp and hair follicles to strengthen and nourish them to prevent hair loss and also to stimulate r

In [4]:
import pandas as pd


In [5]:
response['data']['products']

[{'product_id': 1,
  'name': 'Herforte Lotion Spray (to fill in gaps and treat alopecia)',
  'description': 'Hair forte is a mixture of natural extracts and extracts of the most important hair oils with anti-dandruff agent, to be a quick treatment for hair loss, guaranteed and tested by dermatologists for fortifying hair & increase hair shaft strength to prevent hair loss. Also, Hair forte spray stimulates hair germination to be an effective solution to fill the frontal spaces in women and men, in addition to treating dandruff & itching associated with dryness and inflammation of the scalp, Hair forte spray is suitable for all hair types even oily & dry scalp.\r\n\r\nComposition:-\r\nHair forte spray with its unique formula to be effective in stimulating the blood circulation of the scalp and hair follicles to strengthen and nourish them to prevent hair loss and also to stimulate root germination for the growth of healthy, thick, long and strong hair without dandruff.\r\n\r\n•\tJojoba 

In [6]:
df = pd.DataFrame(response['data']['products'])

In [7]:
df

,product_id,name,description,ingredients,concerns,category,price,product_file,best_seller,new_arrived
0,1,Herforte Lotion Spray (to fill in gaps and tre...,Hair forte is a mixture of natural extracts an...,"[Jojoba Oil Extract, Garlic Oil Extract, Nigel...",[[Hair loss]],"[Hair Care, Hair spray& lotion]",200.00,None,True,False
1,2,Hairforte Shampoo (for treating hair loss and ...,Hair Forte shampoo is an effective combination...,"[Caffeine, Jojoba Oil, Lavender Oil, Coconut O...",[[Dandruff]],"[Hair Care, Shampoos]",150.00,None,True,False
2,3,Valdo moisturizing cream,Valdo Moisturizing Cream\r\n-The best moisturi...,"[Panthenol (Vitamin B5), Aloe Vera, Calendula,...",[[Eczema & Dry skin]],"[Skin Care, Moisturizer]",80.00,None,True,False
3,4,Spleet Cream Scrub (Steam Alternative + Acne T...,Spleet is a gentle exfoliating cream with a un...,"[Urea, Salicylic Acid, Lactic Acid, Tea Tree O...","[[Strawberry skin treatment, Acne& acne spots,...","[Personal Care, Soft peel and Exfoliant]",100.00,None,True,False
4,5,Spleet Lightening Cream (for treating pigmenta...,Spleet whitening cream offers the strongest un...,"[Alpha-Arbutin, Kojic Acid, Niacinamide, Licor...","[[Hyperpigmentation& dark spots, Sensitive are...","[Skin Care, Whitening cream]",100.00,None,True,False
5,6,Spleet Eye Contour Cream (for treating dark ci...,An unique potent mix of natural ingredients...,"[Caffeine, Liquorice Extract, Vitamin A, Green...","[[Hyperpigmentation& dark spots, Dark circles ...","[Skin Care, Eye cream]",225.00,None,True,False
6,7,Spleet Facial wash,Spleet face wash with an unique the Canadian b...,"[Aloe Vera Extract, Jojoba Oil, Tocopherol, Co...","[[Acne& acne spots, Hyperpigmentation& dark sp...","[Skin Care, Cleansers]",220.00,None,True,False
7,12,Aceto-V Feminine Intimate wash,Acet-V is a feminine Intimate wash with unique...,"[Chlorhexidine gluconate, Cetrimide, Aloe Vera...",[[Sensitive areas& Bikini area brightening]],"[Personal Care, Cleansers]",90.00,None,False,False


In [8]:
import pandas as pd



# Function to flatten nested lists (like 'concerns')
def flatten_list(nested_list):
    if isinstance(nested_list, list):
        if not nested_list:
            return []
        elif all(isinstance(item, str) for item in nested_list):
            return nested_list
        else:
            return [item for sublist in nested_list for item in (sublist if isinstance(sublist, list) else [sublist])]
    return nested_list

# Preprocess the DataFrame
def preprocess_dataframe(df):
    # Create a copy to avoid modifying the original
    processed_df = df.copy()
    
    # Handle nested lists
    if 'concerns' in processed_df.columns:
        processed_df['concerns'] = processed_df['concerns'].apply(flatten_list)
        processed_df['concerns'] = processed_df['concerns'].apply(lambda x: ', '.join(x) if isinstance(x, list) else x)
    
    if 'ingredients' in processed_df.columns:
        processed_df['ingredients'] = processed_df['ingredients'].apply(lambda x: ', '.join(x) if isinstance(x, list) else x)
        
    if 'category' in processed_df.columns:
        processed_df['category'] = processed_df['category'].apply(lambda x: ', '.join(x) if isinstance(x, list) else x)
    
    # Clean text fields - remove excessive whitespace and special characters
    text_columns = ['name', 'description']
    for col in text_columns:
        if col in processed_df.columns:
            # Replace multiple whitespaces with a single space
            processed_df[col] = processed_df[col].str.replace(r'\s+', ' ', regex=True)
            # Replace \r\n with space
            processed_df[col] = processed_df[col].str.replace(r'\r\n', ' ', regex=True)
            # Trim whitespace
            processed_df[col] = processed_df[col].str.strip()
    
    # Extract short description (first 100 characters)
    if 'description' in processed_df.columns:
        processed_df['short_description'] = processed_df['description'].apply(
            lambda x: x[:100] + '...' if isinstance(x, str) and len(x) > 100 else x
        )
    
    # Convert price to numeric
    if 'price' in processed_df.columns:
        processed_df['price'] = pd.to_numeric(processed_df['price'], errors='coerce')
    
    # Handle None values
    processed_df = processed_df.fillna(value={
        'product_file': 'No file',
        'ingredients': 'Not specified',
        'concerns': 'Not specified'
    })
    
    # Extract parentheses content from name as a feature
    if 'name' in processed_df.columns:
        processed_df['name_attributes'] = processed_df['name'].str.extract(r'\((.*?)\)', expand=False)
        # Clean up the main name by removing parentheses content
        processed_df['clean_name'] = processed_df['name'].str.replace(r'\s*\(.*?\)\s*', ' ', regex=True).str.strip()
    
    return processed_df

# Apply preprocessing
processed_df = preprocess_dataframe(df)

# Display the preprocessed DataFrame
print("Original DataFrame:")
df.head()


Original DataFrame:


,product_id,name,description,ingredients,concerns,category,price,product_file,best_seller,new_arrived
0,1,Herforte Lotion Spray (to fill in gaps and tre...,Hair forte is a mixture of natural extracts an...,"[Jojoba Oil Extract, Garlic Oil Extract, Nigel...",[[Hair loss]],"[Hair Care, Hair spray& lotion]",200.00,None,True,False
1,2,Hairforte Shampoo (for treating hair loss and ...,Hair Forte shampoo is an effective combination...,"[Caffeine, Jojoba Oil, Lavender Oil, Coconut O...",[[Dandruff]],"[Hair Care, Shampoos]",150.00,None,True,False
2,3,Valdo moisturizing cream,Valdo Moisturizing Cream\r\n-The best moisturi...,"[Panthenol (Vitamin B5), Aloe Vera, Calendula,...",[[Eczema & Dry skin]],"[Skin Care, Moisturizer]",80.00,None,True,False
3,4,Spleet Cream Scrub (Steam Alternative + Acne T...,Spleet is a gentle exfoliating cream with a un...,"[Urea, Salicylic Acid, Lactic Acid, Tea Tree O...","[[Strawberry skin treatment, Acne& acne spots,...","[Personal Care, Soft peel and Exfoliant]",100.00,None,True,False
4,5,Spleet Lightening Cream (for treating pigmenta...,Spleet whitening cream offers the strongest un...,"[Alpha-Arbutin, Kojic Acid, Niacinamide, Licor...","[[Hyperpigmentation& dark spots, Sensitive are...","[Skin Care, Whitening cream]",100.00,None,True,False


In [9]:
print("\nPreprocessed DataFrame:")
processed_df.head()


Preprocessed DataFrame:


,product_id,name,description,ingredients,concerns,category,price,product_file,best_seller,new_arrived,short_description,name_attributes,clean_name
0,1,Herforte Lotion Spray (to fill in gaps and tre...,Hair forte is a mixture of natural extracts an...,"Jojoba Oil Extract, Garlic Oil Extract, Nigell...",Hair loss,"Hair Care, Hair spray& lotion",200.0,No file,True,False,Hair forte is a mixture of natural extracts an...,to fill in gaps and treat alopecia,Herforte Lotion Spray
1,2,Hairforte Shampoo (for treating hair loss and ...,Hair Forte shampoo is an effective combination...,"Caffeine, Jojoba Oil, Lavender Oil, Coconut Oi...",Dandruff,"Hair Care, Shampoos",150.0,No file,True,False,Hair Forte shampoo is an effective combination...,for treating hair loss and dandruff + deep moi...,Hairforte Shampoo
2,3,Valdo moisturizing cream,Valdo Moisturizing Cream -The best moisturizin...,"Panthenol (Vitamin B5), Aloe Vera, Calendula, ...",Eczema & Dry skin,"Skin Care, Moisturizer",80.0,No file,True,False,Valdo Moisturizing Cream -The best moisturizin...,NaN,Valdo moisturizing cream
3,4,Spleet Cream Scrub (Steam Alternative + Acne T...,Spleet is a gentle exfoliating cream with a un...,"Urea, Salicylic Acid, Lactic Acid, Tea Tree Oi...","Strawberry skin treatment, Acne& acne spots, H...","Personal Care, Soft peel and Exfoliant",100.0,No file,True,False,Spleet is a gentle exfoliating cream with a un...,Steam Alternative + Acne Treatment,Spleet Cream Scrub
4,5,Spleet Lightening Cream (for treating pigmenta...,Spleet whitening cream offers the strongest un...,"Alpha-Arbutin, Kojic Acid, Niacinamide, Licori...","Hyperpigmentation& dark spots, Sensitive areas...","Skin Care, Whitening cream",100.0,No file,True,False,Spleet whitening cream offers the strongest un...,for treating pigmentation and acne scars,Spleet Lightening Cream


In [23]:


# Create keyword columns for easier searching
def create_keyword_columns(df):
    # Create a column that combines all searchable text
    df['search_text'] = (
        df['clean_name'].fillna('') + ' ' + 
        df['description'].fillna('') + ' ' + 
        df['ingredients'].fillna('') + ' ' + 
        df['concerns'].fillna('') + ' ' + 
        df['category'].fillna('')
    )
    
    # Convert search text to lowercase for case-insensitive search
    df['search_text'] = df['search_text'].str.lower()
    
    # Create price range categories
    if 'price' in df.columns:
        bins = [0, 50, 100, 200, 500, float('inf')]
        labels = ['0-50', '51-100', '101-200', '201-500', '500+']
        df['price_range'] = pd.cut(df['price'], bins=bins, labels=labels, right=False)
    
    return df

# Apply keyword enhancements
enhanced_df = create_keyword_columns(processed_df)

# Sample queries to demonstrate the functionality
def query_products(df, search_term=None, category=None, concerns=None, price_min=None, price_max=None, has_ingredient=None , best_seller=None , new_arrived=None):
    result = df.copy()
    
    if search_term:
        search_term = search_term.lower()
        result = result[result['search_text'].str.contains(search_term, na=False)]
    
    if category:
        result = result[result['category'].str.contains(category, case=False, na=False)]
        
    if concerns:
        result = result[result['concerns'].str.contains(concerns, case=False, na=False)]
    
    if price_min is not None:
        result = result[result['price'] >= price_min]
        
    if price_max is not None:
        result = result[result['price'] <= price_max]
    
    if has_ingredient:
        result = result[result['ingredients'].str.contains(has_ingredient, case=False, na=False)]
        
    if best_seller:
        result = result[result['best_seller'] == best_seller]
        
    if new_arrived:
        result = result[result['new_arrived'] == new_arrived]
    

    return result[['product_id', 'name', 'description', 'ingredients', 'concerns', 'category' , "price" , "best_seller" , "new_arrived"]]

# Examples of using the query function
print("\nExample Query - Search for 'hair loss':")
result1 = query_products(enhanced_df, search_term='hair Dandruff')
result1




Example Query - Search for 'hair loss':


,product_id,name,description,ingredients,concerns,category,price,best_seller,new_arrived


In [15]:
enhanced_df.iloc[0][-2]

C:\Users\DELL\AppData\Local\Temp\ipykernel_17936\3106559214.py:1: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  enhanced_df.iloc[0][-2]


'herforte lotion spray hair forte is a mixture of natural extracts and extracts of the most important hair oils with anti-dandruff agent, to be a quick treatment for hair loss, guaranteed and tested by dermatologists for fortifying hair & increase hair shaft strength to prevent hair loss. also, hair forte spray stimulates hair germination to be an effective solution to fill the frontal spaces in women and men, in addition to treating dandruff & itching associated with dryness and inflammation of the scalp, hair forte spray is suitable for all hair types even oily & dry scalp. composition:- hair forte spray with its unique formula to be effective in stimulating the blood circulation of the scalp and hair follicles to strengthen and nourish them to prevent hair loss and also to stimulate root germination for the growth of healthy, thick, long and strong hair without dandruff. • jojoba oil extract. • garlic oil extract. • nigella. • rosemary. • caffeine. • thyme oil. • salicylic acid. • p

In [19]:
# Examples of using the query function
result1 = query_products(enhanced_df, search_term=' Care')
result1


,product_id,name,description,ingredients,concerns,category,price,best_seller,new_arrived
0,1,Herforte Lotion Spray (to fill in gaps and tre...,Hair forte is a mixture of natural extracts an...,"Jojoba Oil Extract, Garlic Oil Extract, Nigell...",Hair loss,"Hair Care, Hair spray& lotion",200.0,True,False
1,2,Hairforte Shampoo (for treating hair loss and ...,Hair Forte shampoo is an effective combination...,"Caffeine, Jojoba Oil, Lavender Oil, Coconut Oi...",Dandruff,"Hair Care, Shampoos",150.0,True,False
2,3,Valdo moisturizing cream,Valdo Moisturizing Cream -The best moisturizin...,"Panthenol (Vitamin B5), Aloe Vera, Calendula, ...",Eczema & Dry skin,"Skin Care, Moisturizer",80.0,True,False
3,4,Spleet Cream Scrub (Steam Alternative + Acne T...,Spleet is a gentle exfoliating cream with a un...,"Urea, Salicylic Acid, Lactic Acid, Tea Tree Oi...","Strawberry skin treatment, Acne& acne spots, H...","Personal Care, Soft peel and Exfoliant",100.0,True,False
4,5,Spleet Lightening Cream (for treating pigmenta...,Spleet whitening cream offers the strongest un...,"Alpha-Arbutin, Kojic Acid, Niacinamide, Licori...","Hyperpigmentation& dark spots, Sensitive areas...","Skin Care, Whitening cream",100.0,True,False
5,6,Spleet Eye Contour Cream (for treating dark ci...,An unique potent mix of natural ingredients th...,"Caffeine, Liquorice Extract, Vitamin A, Green ...","Hyperpigmentation& dark spots, Dark circles “b...","Skin Care, Eye cream",225.0,True,False
6,7,Spleet Facial wash,Spleet face wash with an unique the Canadian b...,"Aloe Vera Extract, Jojoba Oil, Tocopherol, Col...","Acne& acne spots, Hyperpigmentation& dark spot...","Skin Care, Cleansers",220.0,True,False
7,12,Aceto-V Feminine Intimate wash,Acet-V is a feminine Intimate wash with unique...,"Chlorhexidine gluconate, Cetrimide, Aloe Vera,...",Sensitive areas& Bikini area brightening,"Personal Care, Cleansers",90.0,False,False


In [69]:
ingredients_set = set()
for ingredients in df['ingredients'].to_list():
    if isinstance(ingredients, list):
        for ingredient in ingredients:
            if isinstance(ingredient, (list, tuple)) and ingredient:
                ingredients_set.add(ingredient[0])  # Add first element if it's a tuple/list
            elif ingredient:  # Add the ingredient if it's a non-empty string
                ingredients_set.add(ingredient)
print(ingredients_set)

{'Vitamin C', 'Tocopherol', 'Chamomile Extract', 'Panthenol', 'Zinc Pyrithione', 'Dimethicone', 'Vitamin E', 'Aloe Vera Extract', 'Salicylic acid', 'Jojoba Oil Extract', 'Glycolic Acid', 'Glycine', 'Jojoba Oil', 'Royal Jelly', 'Chlorhexidine gluconate', 'Chamomile', 'Zinc Oxide', 'Olive Oil', 'Licorice', 'Green Tea', 'Triclosan', 'Panthenol (Vitamin B5)', 'Urea', 'Vitamin A', 'Wheat Germ Oil', 'Cetrimide', 'Calendula', 'Nigella', 'Rosemary Oil', 'Salicylic Acid', 'Glycerin', 'Collagen', 'Lavender Oil', 'Coconut Oil', 'Garlic Oil Extract', 'Rosemary', 'Kojic Acid', 'Glycyrrhiza Glabra Leaf Extract', 'Garlic Extract', 'Aloe Vera', 'Allantoin', 'Niacinamide', 'Caffeine', 'Almond Oil', 'Alpha-Arbutin', 'Camellia Sinensis Extract', 'Lactic Acid', 'Thyme Oil', 'Tea Tree Oil', 'Liquorice Extract'}


In [67]:
query_products(enhanced_df, best_seller = True , category='Hair ')

,product_id,name,description,ingredients,concerns,category,price,best_seller,new_arrived
0,1,Herforte Lotion Spray (to fill in gaps and tre...,Hair forte is a mixture of natural extracts an...,"Jojoba Oil Extract, Garlic Oil Extract, Nigell...",Hair loss,"Hair Care, Hair spray& lotion",200.0,True,False
1,2,Hairforte Shampoo (for treating hair loss and ...,Hair Forte shampoo is an effective combination...,"Caffeine, Jojoba Oil, Lavender Oil, Coconut Oi...",Dandruff,"Hair Care, Shampoos",150.0,True,False


In [49]:
print("\nExample Query - Products with price less than 250:")
result2 = query_products(enhanced_df, price_max=250)
result2


Example Query - Products with price less than 250:


,product_id,clean_name,short_description,price,category,ingredients
0,1,Herforte Lotion Spray,Hair forte is a mixture of natural extracts an...,200.0,"Hair Care, Hair spray& lotion","Jojoba Oil Extract, Garlic Oil Extract, Nigell..."
1,2,Hairforte Shampoo,Hair Forte shampoo is an effective combination...,150.0,"Hair Care, Shampoos","Caffeine, Jojoba Oil, Lavender Oil, Coconut Oi..."
2,3,Valdo moisturizing cream,Valdo Moisturizing Cream -The best moisturizin...,80.0,"Skin Care, Moisturizer","Panthenol (Vitamin B5), Aloe Vera, Calendula, ..."
3,4,Spleet Cream Scrub,Spleet is a gentle exfoliating cream with a un...,100.0,"Personal Care, Soft peel and Exfoliant","Urea, Salicylic Acid, Lactic Acid, Tea Tree Oi..."
4,5,Spleet Lightening Cream,Spleet whitening cream offers the strongest un...,100.0,"Skin Care, Whitening cream","Alpha-Arbutin, Kojic Acid, Niacinamide, Licori..."
5,6,Spleet Eye Contour Cream,An unique potent mix of natural ingredients th...,225.0,"Skin Care, Eye cream","Caffeine, Liquorice Extract, Vitamin A, Green ..."
6,7,Spleet Facial wash,Spleet face wash with an unique the Canadian b...,220.0,"Skin Care, Cleansers","Aloe Vera Extract, Jojoba Oil, Tocopherol, Col..."
7,12,Aceto-V Feminine Intimate wash,Acet-V is a feminine Intimate wash with unique...,90.0,"Personal Care, Cleansers","Chlorhexidine gluconate, Cetrimide, Aloe Vera,..."


In [53]:


print("\nExample Query - Products with Aloe Vera:")
result3 = query_products(enhanced_df, has_ingredient='Jojoba')
result3






Example Query - Products with Aloe Vera:


,product_id,clean_name,short_description,price,category,ingredients
0,1,Herforte Lotion Spray,Hair forte is a mixture of natural extracts an...,200.0,"Hair Care, Hair spray& lotion","Jojoba Oil Extract, Garlic Oil Extract, Nigell..."
1,2,Hairforte Shampoo,Hair Forte shampoo is an effective combination...,150.0,"Hair Care, Shampoos","Caffeine, Jojoba Oil, Lavender Oil, Coconut Oi..."
6,7,Spleet Facial wash,Spleet face wash with an unique the Canadian b...,220.0,"Skin Care, Cleansers","Aloe Vera Extract, Jojoba Oil, Tocopherol, Col..."


In [48]:
enhanced_df

,product_id,name,description,ingredients,concerns,category,price,product_file,best_seller,new_arrived,short_description,name_attributes,clean_name,search_text,price_range
0,1,Herforte Lotion Spray (to fill in gaps and tre...,Hair forte is a mixture of natural extracts an...,"Jojoba Oil Extract, Garlic Oil Extract, Nigell...",Hair loss,"Hair Care, Hair spray& lotion",200.0,No file,True,False,Hair forte is a mixture of natural extracts an...,to fill in gaps and treat alopecia,Herforte Lotion Spray,herforte lotion spray hair forte is a mixture ...,201-500
1,2,Hairforte Shampoo (for treating hair loss and ...,Hair Forte shampoo is an effective combination...,"Caffeine, Jojoba Oil, Lavender Oil, Coconut Oi...",Dandruff,"Hair Care, Shampoos",150.0,No file,True,False,Hair Forte shampoo is an effective combination...,for treating hair loss and dandruff + deep moi...,Hairforte Shampoo,hairforte shampoo hair forte shampoo is an eff...,101-200
2,3,Valdo moisturizing cream,Valdo Moisturizing Cream -The best moisturizin...,"Panthenol (Vitamin B5), Aloe Vera, Calendula, ...",Eczema & Dry skin,"Skin Care, Moisturizer",80.0,No file,True,False,Valdo Moisturizing Cream -The best moisturizin...,NaN,Valdo moisturizing cream,valdo moisturizing cream valdo moisturizing cr...,51-100
3,4,Spleet Cream Scrub (Steam Alternative + Acne T...,Spleet is a gentle exfoliating cream with a un...,"Urea, Salicylic Acid, Lactic Acid, Tea Tree Oi...","Strawberry skin treatment, Acne& acne spots, H...","Personal Care, Soft peel and Exfoliant",100.0,No file,True,False,Spleet is a gentle exfoliating cream with a un...,Steam Alternative + Acne Treatment,Spleet Cream Scrub,spleet cream scrub spleet is a gentle exfoliat...,101-200
4,5,Spleet Lightening Cream (for treating pigmenta...,Spleet whitening cream offers the strongest un...,"Alpha-Arbutin, Kojic Acid, Niacinamide, Licori...","Hyperpigmentation& dark spots, Sensitive areas...","Skin Care, Whitening cream",100.0,No file,True,False,Spleet whitening cream offers the strongest un...,for treating pigmentation and acne scars,Spleet Lightening Cream,spleet lightening cream spleet whitening cream...,101-200
5,6,Spleet Eye Contour Cream (for treating dark ci...,An unique potent mix of natural ingredients th...,"Caffeine, Liquorice Extract, Vitamin A, Green ...","Hyperpigmentation& dark spots, Dark circles “b...","Skin Care, Eye cream",225.0,No file,True,False,An unique potent mix of natural ingredients th...,for treating dark circles + expression lines +...,Spleet Eye Contour Cream,spleet eye contour cream an unique potent mix ...,201-500
6,7,Spleet Facial wash,Spleet face wash with an unique the Canadian b...,"Aloe Vera Extract, Jojoba Oil, Tocopherol, Col...","Acne& acne spots, Hyperpigmentation& dark spot...","Skin Care, Cleansers",220.0,No file,True,False,Spleet face wash with an unique the Canadian b...,NaN,Spleet Facial wash,spleet facial wash spleet face wash with an un...,201-500
7,12,Aceto-V Feminine Intimate wash,Acet-V is a feminine Intimate wash with unique...,"Chlorhexidine gluconate, Cetrimide, Aloe Vera,...",Sensitive areas& Bikini area brightening,"Personal Care, Cleansers",90.0,No file,False,False,Acet-V is a feminine Intimate wash with unique...,NaN,Aceto-V Feminine Intimate wash,aceto-v feminine intimate wash acet-v is a fem...,51-100


In [18]:
class DfManager():
    
    def __init__(self, df):
        self.df = df
        self.__clean_df()
        
    def __clean_df(self):
        self.df['price'] = self.df['price'].astype(float)
        return self.df
        
    def get_df(self):
        return self.df

    def get_category_set(self):
        category_set = set()
        for category in self.df['category'].to_list():  
            for c in category:
                category_set.add(c)
        return category_set

    def get_concerns_set(self):
        concerns_set = set()
        for concerns in self.df['concerns'].to_list():
            for concern in concerns:
                for c in concern:
                    concerns_set.add(c)
        return concerns_set


In [19]:
df_manager = DfManager(df)

In [24]:
df_manager.get_df()

,product_id,name,description,ingredients,concerns,category,price,product_file
0,1,Herforte Lotion Spray (to fill in gaps and tre...,Hair forte is a mixture of natural extracts an...,[],[[Hair loss]],"[Hair Care, Hair spray& lotion]",200.0,None
1,2,Hairforte Shampoo (for treating hair loss and ...,Hair Forte shampoo is an effective combination...,[],[[Dandruff]],"[Hair Care, Shampoos]",150.0,None
2,3,Valdo moisturizing cream,Valdo Moisturizing Cream\r\n-The best moisturi...,[],[[Eczema & Dry skin]],"[Skin Care, Moisturizer]",80.0,None
3,4,Spleet Cream Scrub (Steam Alternative + Acne T...,Spleet is a gentle exfoliating cream with a un...,[],"[[Strawberry skin treatment, Acne& acne spots,...","[Personal Care, Soft peel and Exfoliant]",100.0,None
4,5,Spleet Lightening Cream (for treating pigmenta...,Spleet whitening cream offers the strongest un...,[],"[[Hyperpigmentation& dark spots, Sensitive are...","[Skin Care, Whitening cream]",100.0,None
5,6,Spleet Eye Contour Cream (for treating dark ci...,An unique potent mix of natural ingredients...,[],"[[Hyperpigmentation& dark spots, Dark circles ...","[Skin Care, Eye cream]",225.0,None
6,7,Spleet Facial wash,Spleet face wash with an unique the Canadian b...,[],"[[Acne& acne spots, Hyperpigmentation& dark sp...","[Skin Care, Cleansers]",220.0,None
7,12,Aceto-V Feminine Intimate wash,Acet-V is a feminine Intimate wash with unique...,[],[[Sensitive areas& Bikini area brightening]],"[Personal Care, Cleansers]",90.0,None
8,14,"Valdo Sunscreen Lotion (No Oxidation, No Sweat...",الحمايةمن االلتهاباتوجفاف البشرة.\r\nيحمي من...,[],[[Eczema & Dry skin]],"[Skin Care, Sun screen]",350.0,None
9,15,Hairforte Oil Replacement Cream (for treating ...,Hairfort cream is a special formula composed...,[],[[Dandruff]],"[Hair Care, Oil replacement]",180.0,None


In [7]:
category_set = set()

for category in df['category'].to_list():
  for c in category:
    category_set.add(c)

category_set

{'Cleansers',
 'Eye cream',
 'Hair Care',
 'Hair spray& lotion',
 'Moisturizer',
 'Oil replacement',
 'Personal Care',
 'Repairing & hair moisturizing',
 'Shampoos',
 'Skin Care',
 'Soft peel and Exfoliant',
 'Sun screen',
 'Toner “coming soon”',
 'Whitening cream'}

In [8]:
concerns_set = set()

for concerns in df['concerns'].to_list():
    for concern in concerns:
        for c in concern:
            concerns_set.add(c)

concerns_set

{'Acne& acne spots',
 'Dandruff',
 'Dark circles “blue- brown- black” & puffiness',
 'Dull& pale skin',
 'Eczema & Dry skin',
 'Hair loss',
 'Hyperpigmentation& dark spots',
 'Sensitive areas& Bikini area brightening',
 'Strawberry skin treatment'}

In [10]:


# Filter by categories
categories = ['Hair Care', 'Shampoos']
concerns = ['Hair loss', 'Dandruff']

category_matches = df[df['category'].apply(
        lambda x: any(cat.lower() in c.lower() for cat in categories for c in x)
    )]
    
# Further filter by concerns
matches = category_matches[category_matches['concerns'].apply(
        lambda x: any(con.lower() in c[0].lower() for con in concerns for c in x if c)
    )]

In [12]:
matches.to_dict('records')

[{'product_id': 1,
  'name': 'Herforte Lotion Spray (to fill in gaps and treat alopecia)',
  'description': 'Hair forte is a mixture of natural extracts and extracts of the most important hair oils with anti-dandruff agent, to be a quick treatment for hair loss, guaranteed and tested by dermatologists for fortifying hair & increase hair shaft strength to prevent hair loss. Also, Hair forte spray stimulates hair germination to be an effective solution to fill the frontal spaces in women and men, in addition to treating dandruff & itching associated with dryness and inflammation of the scalp, Hair forte spray is suitable for all hair types even oily & dry scalp.\r\n\r\nComposition:-\r\nHair forte spray with its unique formula to be effective in stimulating the blood circulation of the scalp and hair follicles to strengthen and nourish them to prevent hair loss and also to stimulate root germination for the growth of healthy, thick, long and strong hair without dandruff.\r\n\r\n•\tJojoba 

In [ ]:
@tool
def filter_products_by_category_and_concern(categories: list, concerns: list) -> List[Dict]:
    """
    Filter products by both categories and concerns.
    
    Args:
        categories (list): List of product categories to filter by (e.g., ['Hair Care', 'Shampoos'])
        concerns (list): List of health/beauty concerns to filter by (e.g., ['Hair loss', 'Dandruff'])
    
    Returns:
        List[Dict]: A list of products matching both the specified categories and concerns
    """
    # Filter by categories
    category_matches = df[df['category'].apply(
        lambda x: any(cat.lower() in c.lower() for cat in categories for c in x)
    )]
    
    # Further filter by concerns
    matches = category_matches[category_matches['concerns'].apply(
        lambda x: any(con.lower() in c[0].lower() for con in concerns for c in x if c)
    )]
    
    return matches.to_dict('records')

In [9]:
from typing import List, Dict, Optional
from langchain_core.tools import tool
import pandas as pd

# Assuming your DataFrame is loaded as df
# df = pd.read_csv('your_products.csv')

@tool
def search_products_by_concern(concern: str) -> List[Dict]:
    """
    Search for products by health/beauty concern (e.g., 'Hair loss', 'Dandruff', 'Eczema').
    
    Args:
        concern (str): The health or beauty concern to search for
    
    Returns:
        List[Dict]: A list of products addressing the specified concern
    """
    matches = df[df['concerns'].apply(
        lambda x: any(concern.lower() in c[0].lower() for c in x if c)
    )]
    return matches.to_dict('records')

@tool
def search_products_by_category(category: str) -> List[Dict]:
    """
    Search for products by category (e.g., 'Hair Care', 'Skin Care').
    
    Args:
        category (str): The category to search for
    
    Returns:
        List[Dict]: A list of products in the specified category
    """
    matches = df[df['category'].apply(
        lambda x: any(category.lower() in c.lower() for c in x)
    )]
    return matches.to_dict('records')

@tool
def search_products_by_name(name: str) -> List[Dict]:
    """
    Search for products by name.
    
    Args:
        name (str): The product name to search for (partial matches allowed)
    
    Returns:
        List[Dict]: A list of products matching the name query
    """
    matches = df[df['name'].str.lower().str.contains(name.lower())]
    return matches.to_dict('records')

@tool
def get_lowest_price_products(n: int = 5) -> List[Dict]:
    """
    Get the lowest priced products.
    
    Args:
        n (int): Number of products to return
    
    Returns:
        List[Dict]: A list of the n lowest priced products
    """
    return df.nsmallest(n, 'price').to_dict('records')

@tool
def get_highest_price_products(n: int = 5) -> List[Dict]:
    """
    Get the highest priced products.
    
    Args:
        n (int): Number of products to return
    
    Returns:
        List[Dict]: A list of the n highest priced products
    """
    return df.nlargest(n, 'price').to_dict('records')

@tool
def get_products_in_price_range(min_price: float, max_price: float) -> List[Dict]:
    """
    Get products within a specific price range.
    
    Args:
        min_price (float): Minimum price
        max_price (float): Maximum price
    
    Returns:
        List[Dict]: List of products within the specified price range
    """
    matches = df[(df['price'] >= min_price) & (df['price'] <= max_price)]
    return matches.to_dict('records')

# List of tools to be used with LangGraph
tools = [
    search_products_by_concern,
    search_products_by_category,
    search_products_by_name,
    get_lowest_price_products,
    get_highest_price_products,
    get_products_in_price_range
]

In [ ]:
search_products_by_concern()